# Step 2: Profiling LLM Inference

In this step, we shift from image processing to a more contemporary workload: Large Language Model (LLM) inference. We'll profile TinyLlama-1.1B to understand the memory access patterns that occur during text generation. This is particularly interesting because LLM inference has a very different performance profile than the compute-bound kernel we saw in Step 1.

## Transformer LLM Architecture

Modern LLMs are built on the **Transformer** architecture. Here is a conceptual overview. Note that depending on the use case, the LLM may use only encoders, only decoders, or both.

<img src="images/encoder_decoder.png" width="50%"/>

<table style="margin-left: 0;">
    <tr><td><b>Self-Attention</b></td><td>Allows each token to "look at" all previous tokens</td><td>Matrix multiplications + softmax</td></tr>
    <tr><td><b>Feed-Forward</b></td><td>Per-token nonlinear transformation</td><td>Two dense layers with activation</td></tr>
</table>

### Self-Attention Mechanism

The core innovation is **self-attention**, which computes relationships between all tokens ([unmasked](https://stackoverflow.blog/2024/09/26/masked-self-attention-how-llms-learn-relationships-between-tokens/) variant is shown):
   
$\displaystyle $$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$

- **Q**uery: "What am I looking for?"
- **K**ey: "What do I contain?"
- **V**alue: "What information do I provide?"

The attention score between tokens $i$ and $j$ is $\frac{Q_i \cdot K_j}{\sqrt{d_k}}$. Softmax normalizes these scores to sum to 1, and the output is a weighted sum of Values.

Multi-Head Attention runs multiple attention operations in parallel (e.g., 32 heads), each learning different relationships.

### Components Relevant to This Lab

For our profiling exercise, we focus on the following mechanisms:

<table style="margin-left: 0;">
    <tr><th>Component</th><th>Description</th></tr>
    <tr><td><b>KV-Cache</b></td><td>Stores computed Key and Value tensors to avoid recomputation</td></tr>
    <tr><td><b>Prefill</b></td><td>Process all input tokens at once, build initial KV-cache</td></tr>
    <tr><td><b>Decode</b></td><td>Generate tokens one at a time, reading entire KV-cache each step</td></tr>
</table>


<img src="images/prefill.png" width="30%"/>

During **prefill**, the model processes all input tokens in parallel and builds the initial KV-cache. Because we have access to the entire input prompt at once, we can compute attention for all positions simultaneously. This phase involves:

- Computing Q, K, V projections for every input token in one batched matrix multiplication
- Computing attention scores (Q @ Kᵀ) across the full sequence
- Storing K and V tensors in the KV-cache for reuse during decode

Prefill is typically **compute-bound** in optimized kernels: the large batch size (sequence length) enables high arithmetic intensity, allowing the GPU to fully utilize its tensor cores. Memory bandwidth is less of a concern because we read model weights once and apply them to many tokens.

<img src="images/decode.png" width="30%"/>

During **decode**, the model generates one token at a time. This means that for each new token generated, we must read all the Keys and Values from every previous token. As the sequence gets longer, this memory traffic grows linearly, making this **memory bound**. This is the fundamental bottleneck that techniques like Flash Attention, paged attention, and KV-cache quantization aim to address. Such kernels can create a pattern of **Long and Short Scoreboard stalls** for unoptimized kernel, that we can observe in the profiler.

## PyTorch LLM Inference Example 

[PyTorch](https://pytorch.org/) is a popular deep learning framework. It provides high-level tensor operations that dispatch to optimized CUDA kernels (cuBLAS, cuDNN, or custom ATen kernels). PyTorch operations ultimately call CUDA kernels. Nsight Compute profiles whatever CUDA kernels run on the GPU, regardless of the host language.

```python
output = torch.matmul(Q, K.transpose(-2, -1))  # cuBLAS GEMM
output = F.softmax(output, dim=-1)             # ATen kernel
```

[decode.py](source/decode.py) implements our inference run. We're using TinyLlama-1.1B, a smaller model that still exhibits similar memory access patterns as larger LLMs like LLaMA-70B — just at a more manageable scale for profiling in this lab. Run the cell below to execute the script without Nsight Compute. While it's running, we discuss the structure.

In [ ]:
!python3 source/decode.py

### Configuration

- **Larger batch sizes** increase parallelism but also increase KV-cache memory traffic proportionally
- **Longer sequences** (more tokens generated) make the memory bottleneck more pronounced as the KV-cache grows
- **Longer prompts** start with a larger initial KV-cache, affecting all subsequent decode steps

```Python
MAX_NEW_TOKENS = 1000   # Number of tokens to generate
BATCH_SIZE = 4          # Number of parallel sequences
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
```

### Setup

Next we load the model and tokenizer. There are a few important configuration choices here:

- `dtype=torch.float16` — FP16 precision, standard for inference, balances performance and memory usage
- `attn_implementation="eager"` — This forces PyTorch to use standard attention kernels instead of FlashAttention. While FlashAttention is faster in practice, it fuses multiple operations into a single optimized kernel, which makes it harder to see the individual attention operations in the profiler. Using eager mode lets us observe the raw attention behavior for learning purposes.

```Python
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Load model in FP16 with standard attention
model = AutoModelForCausalLM.from_pretrained(...)
```

After loading, we'll see the model's parameter count and verify it's on the GPU.

### Prefill

In the **prefill** phase, the model processes all input tokens in parallel and builds the initial KV-cache.
We are using [NVTX](https://github.com/NVIDIA/NVTX), NVIDIA's tools extension API, to mark the prefill phase with an NVTX Push/Pop range. This later allows our tools to be aware of it for filtering and output.

After prefill completes, we'll have:
- The initial KV-cache populated with Keys and Values for all prompt tokens
- The logits for predicting the first new token
- A baseline for how large the KV-cache starts (which affects all subsequent decode steps)

```Python
# Process input and build KV-cache
import nvtx
with nvtx.annotate("Prefill"):
    with torch.no_grad():
        outputs = model(...)

# Show KV-cache info
...
```

### Decode

The decode loop is where we generate new tokens one at a time. Each iteration of the loop:
1. Samples the next token from the logits
2. Runs a forward pass with only that single new token
3. Reads the **entire** KV-cache to compute attention
4. Updates the KV-cache with the new token's K and V vectors
5. Is marked with a unique NVTX range name

```Python
for step in range(MAX_NEW_TOKENS):
    with nvtx.annotate(f"DecodeStep{step}"):
        # Sample next token
        ...

        # Check for EOS
        ...

        # Forward pass with KV-cache (only the new token)
        ...
```

### Output

Lastly, we print the generated text and final KV-cache size.

## Profiling

We now want to profile our inference code using the Nsight Compute command line interface, `ncu`.
We prepared the entire code as a standalone script at [decode.py](source/decode.py).
Note that we are using [NVTX](https://github.com/NVIDIA/NVTX), NVIDIA's tools extension API, to mark ranges of interest with NVTX Push/Pop ranges to select a subset of the kernels for profiling.

<table style="margin-left: 0;">
    <tr><td><b>--nvtx</b></td><td>Enable NVTX tracking</td></tr>
    <tr><td><b>--nvtx-include "Prefill/"</b></td><td>Include kernels in the Push/Pop NVTX range "Prefill"</td></tr>
    <tr><td><b>--kernel-id :::1</b></td><td>Profile the first instance of each unique kernel</td></tr>
    <tr><td><b>--print-kernel-base function</b></td><td>Simplified function name printed</td></tr>
    <tr><td><b>(others)</b></td><td>For making the output more convenient in this lab environment</td></tr>
</table>

Run the below notebook cell to profile it inspect the printed results.

In [ ]:
!ncu --nvtx --nvtx-include "Prefill/" --nvtx-include "DecodeStep150/" \
  --kernel-id :::1  \
  --metrics sm__throughput.avg.pct_of_peak_sustained_elapsed,gpu__compute_memory_throughput.avg.pct_of_peak_sustained_elapsed,gpu__time_duration.sum \
  -f -o reports/kvcache \
  --page details \
  --print-kernel-base function \
  --import-source yes \
  python3 source/decode.py --max-tokens 200

### What do we see

- The `nvjet_sm90_hsh` kernels take the most time
- Some have very high compute and memory throughput ~80%, indicating either already great optimization, or potential for further improvements by reducing bottlenecks
- It's used for both prefill and decode
- The longest one, by far, is in the KV-Cache *prefill* range - prefill looks at all tokens together, while decode only calculates the next token

Looking it up, `nvjet_sm90_hsh` is NVIDIA's JIT-compiled efficient *transformer/attention* kernel for Hopper (SM 90).

Let's try to profile a standalone attention kernel in more detail to learn more about using Nsight Compute.

---

## Summary

<div class="alert alert-block alert-success">
    <p><b>LLM inference has two phases</b>: <i>prefill</i> (parallel, normally compute-bound) and <i>decode</i> (sequential, normally memory-bound).</p>
    <p><b>KV-cache</b> stores Keys and Values to avoid recomputation, but reading it grows linearly with sequence length.</p>
    <p><b>NVTX markers</b> let you isolate specific code regions (e.g., "Prefill", "DecodeStep150") for targeted profiling.</p>
    <p><b>Nsight Compute</b> reveals that attention kernels dominate runtime, with prefill attention being the longest single kernel invocation in this example.</p>
</div>

**Next: [Step 3 - Optimizing Triton Attention Kernels](step3.ipynb)**